# 6.7 — SGD, Momentum & Nesterov

SGD turns gradients into parameter movement, Momentum adds velocity so repeated evidence becomes motion, and Nesterov checks the gradient at the lookahead point before committing. In this lesson, you will build every optimizer update from NumPy arrays, watch the paths they take on simple objectives, and see why scale, noise, and memory bookkeeping matter in deep learning.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build SGD, Momentum, and Nesterov one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is exposed so the optimizers feel like arithmetic, not magic. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, gradients, vectorized optimizer paths.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for noisy gradients and demos.

### 1. A tiny forward pass creates the signal an optimizer must move

Deep learning optimization begins after a model produces a score and a loss produces a gradient. To keep the arithmetic visible, start with two inputs, one affine unit, a ReLU gate, and a softmax comparison against a baseline. The lesson's scratch numbers are small enough to inspect by hand: $x=[1.5,-0.5]$, $w=[1.4,0.2]$, and $b=0.4$.

In [ ]:
x_w = np.array([1.5, -0.5])            # two input features.
w_w = np.array([1.4, 0.2])             # one weight per feature.
b_w = 0.4                              # scalar bias.
pieces_w = w_w * x_w                   # per-feature affine contributions.
z_w = float(np.sum(pieces_w) + b_w)     # affine score before gating.
print("affine pieces:", pieces_w, "+ bias", b_w)
print("z =", round(z_w, 3))
assert round(z_w, 3) == 2.400

▶ What you'll see: the two feature contributions are `2.1` and `-0.1`, and adding the bias gives the score `2.4`.

In [ ]:
a_w = max(0.0, z_w)                    # ReLU keeps positive signal and clips negative signal.
baseline_w = 0.4                       # comparison score for a second class/action.
exp_scores_w = np.exp([a_w, baseline_w])  # exponentiate scores before normalizing.
prob_w = float(exp_scores_w[0] / exp_scores_w.sum())  # softmax probability for the first score.
print("ReLU output:", round(a_w, 3))
print("softmax probability:", round(prob_w, 3))
assert round(prob_w, 3) == 0.881

▶ What you'll see: the positive score survives the gate and becomes a large softmax probability, about `0.881`.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["affine z", "ReLU a", "baseline", "prob"], [z_w, a_w, baseline_w, prob_w], color=["gray", "teal", "orange", "purple"])
plt.title("1: signal before the optimizer")
plt.ylabel("value")
plt.show()

▶ What you'll see: the raw score is shaped by the gate and then interpreted relative to a baseline.

*Why it's done this way:* optimizers do not see examples directly; they see gradients produced by differentiable computations. The affine sum preserves signed evidence, ReLU keeps only active positive signal, and softmax converts score differences into a comparison that a loss can differentiate. If these local numbers are badly scaled, the gradient that reaches SGD is already distorted.

### 2. Vanilla SGD is one reliable nudge at a time

For one scalar parameter, gradient descent is just $\theta_t=\theta_{t-1}-\eta g_t$. The gradient points uphill in loss, so subtracting it moves downhill. With $\theta=2.0$, learning rate $\eta=0.07$, and gradient $g=1.2$, the lesson's single update is deliberately small.

In [ ]:
theta_w = 2.0                         # current scalar parameter.
eta_w = 0.07                          # learning rate: step size per gradient unit.
g_w = 1.2                             # current gradient dL/dtheta.
step_w = eta_w * g_w                  # amount to subtract from theta.
theta_next_w = theta_w - step_w       # vanilla SGD update.
print("step eta*g:", round(step_w, 3))
print("theta after SGD:", round(theta_next_w, 3))
assert round(theta_next_w, 3) == 1.916

▶ What you'll see: the parameter moves from `2.000` to `1.916`, a small downhill nudge.

In [ ]:
theta_grid_w = np.linspace(-1, 3, 200)             # parameter values for a quadratic loss.
loss_grid_w = (theta_grid_w - 0.5) ** 2            # simple bowl with minimum at 0.5.
loss_before_w = (theta_w - 0.5) ** 2               # loss before one step.
loss_after_w = (theta_next_w - 0.5) ** 2           # loss after one step.
print("loss before -> after:", round(loss_before_w, 3), "->", round(loss_after_w, 3))
assert loss_after_w < loss_before_w

▶ What you'll see: on this bowl, the update lowers the loss because the gradient sign was correct.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(theta_grid_w, loss_grid_w, color="navy")
plt.scatter([theta_w, theta_next_w], [loss_before_w, loss_after_w], color=["red", "green"], zorder=3)
plt.arrow(theta_w, loss_before_w, theta_next_w - theta_w, loss_after_w - loss_before_w, length_includes_head=True, head_width=0.05, color="black")
plt.title("2: one SGD step on a loss bowl")
plt.xlabel("theta")
plt.ylabel("loss")
plt.show()

▶ What you'll see: the point slides a short distance down the quadratic curve.

*Why it's done this way:* the learning rate turns a local slope into a distance. A small $\eta$ trusts the gradient only a little, so learning requires repetition; a large $\eta$ can cross the bottom and increase loss. SGD's power is not one heroic jump — it is many cheap, directionally useful nudges.

### 3. Minibatch gradients are noisy estimates of the full gradient

A full-data gradient is often too expensive, so SGD uses minibatches. Each minibatch gradient is an estimate: cheaper and noisier. The mean direction can still be right, but individual steps zig-zag because different minibatches emphasize different examples.

In [ ]:
true_grad_w = np.array([1.0, 0.15])                    # full-data gradient direction.
noise_w = np.array([[0.30, -0.55], [-0.20, 0.35], [0.15, -0.25], [-0.10, 0.20]])  # minibatch deviations.
mini_grads_w = true_grad_w + noise_w                   # four noisy gradient estimates.
mean_grad_w = mini_grads_w.mean(axis=0)                 # average estimate across minibatches.
print("minibatch gradients:\n", np.round(mini_grads_w, 3))
print("mean minibatch gradient:", np.round(mean_grad_w, 3))
assert np.allclose(mean_grad_w, true_grad_w + noise_w.mean(axis=0))

▶ What you'll see: every minibatch points roughly rightward, but their vertical components disagree.

In [ ]:
theta_path_w = [np.array([2.0, 1.0])]                  # start away from the origin.
eta_noise_w = 0.25                                     # step size for visible movement.
for grad_w in mini_grads_w:                            # apply one SGD step per minibatch.
    theta_path_w.append(theta_path_w[-1] - eta_noise_w * grad_w)
theta_path_w = np.array(theta_path_w)
print("path points:\n", np.round(theta_path_w, 3))
assert theta_path_w.shape == (5, 2)

▶ What you'll see: the parameter moves mostly left but jitters up and down.

In [ ]:
plt.figure(figsize=(4.6, 3.5))
plt.plot(theta_path_w[:, 0], theta_path_w[:, 1], marker="o", color="crimson")
for k_w in range(len(mini_grads_w)):
    plt.text(theta_path_w[k_w, 0], theta_path_w[k_w, 1], str(k_w))
plt.title("3: noisy minibatch SGD path")
plt.xlabel("theta[0]")
plt.ylabel("theta[1]")
plt.show()

▶ What you'll see: the trajectory makes progress in the average direction while bouncing because each minibatch gradient is different.

*Why it's done this way:* minibatches trade exactness for speed. If the noise is roughly unbiased, repeated updates average out and training progresses; if the noise or scale is biased, the local step can repeatedly push the global trajectory away from the desired solution.

### 4. Momentum turns repeated gradients into velocity

Momentum keeps a running velocity: $v_t=\mu v_{t-1}-\eta g_t$, then $\theta_t=\theta_{t-1}+v_t$. Repeated gradients in the same direction accumulate, while alternating noisy components cancel. The parameter now moves with memory instead of starting from rest each step.

In [ ]:
mu_w = 0.9                                      # momentum coefficient: how much velocity persists.
eta_m_w = 0.1                                   # learning rate inside the velocity update.
grads_m_w = np.array([[1.0, 0.6], [1.0, -0.5], [1.0, 0.4], [1.0, -0.3]])  # steady x, noisy y.
v_w = np.zeros(2)                               # start with no motion.
theta_m_w = np.array([2.0, 1.0])                # starting parameter.
velocities_w = []                               # record velocity after each step.
for grad_w in grads_m_w:
    v_w = mu_w * v_w - eta_m_w * grad_w          # velocity remembers old motion and adds new downhill push.
    theta_m_w = theta_m_w + v_w                  # parameter follows velocity.
    velocities_w.append(v_w.copy())
velocities_w = np.array(velocities_w)
print("velocities:\n", np.round(velocities_w, 3))
assert np.allclose(np.round(velocities_w[-1], 3), [-0.344, -0.009])

▶ What you'll see: the x-velocity grows steadily negative, while the y-velocity becomes small because the y-gradients alternate.

In [ ]:
sgd_path_m_w = [np.array([2.0, 1.0])]            # plain SGD path for comparison.
mom_path_w = [np.array([2.0, 1.0])]              # momentum path for comparison.
v_cmp_w = np.zeros(2)
for grad_w in grads_m_w:
    sgd_path_m_w.append(sgd_path_m_w[-1] - eta_m_w * grad_w)
    v_cmp_w = mu_w * v_cmp_w - eta_m_w * grad_w
    mom_path_w.append(mom_path_w[-1] + v_cmp_w)
sgd_path_m_w, mom_path_w = np.array(sgd_path_m_w), np.array(mom_path_w)
print("plain final:", np.round(sgd_path_m_w[-1], 3), "momentum final:", np.round(mom_path_w[-1], 3))
assert mom_path_w[-1, 0] < sgd_path_m_w[-1, 0]

▶ What you'll see: momentum moves farther along the consistently useful x-direction.

In [ ]:
plt.figure(figsize=(4.8, 3.5))
plt.plot(sgd_path_m_w[:, 0], sgd_path_m_w[:, 1], marker="o", label="plain SGD", color="gray")
plt.plot(mom_path_w[:, 0], mom_path_w[:, 1], marker="o", label="momentum", color="teal")
plt.title("4: momentum smooths zig-zag")
plt.xlabel("theta[0]")
plt.ylabel("theta[1]")
plt.legend()
plt.show()

▶ What you'll see: the momentum path accelerates in the persistent direction and is less distracted by alternating vertical noise.

*Why it's done this way:* $\mu v_{t-1}$ is memory. Coordinates whose gradients keep the same sign build speed; coordinates whose gradients flip sign subtract from their own velocity. This is why momentum can move quickly down long valleys while damping side-to-side oscillation.

### 5. Nesterov looks ahead before measuring the gradient

Classical momentum computes the gradient at the current parameter. Nesterov first asks, "where will my velocity carry me?" and evaluates the gradient at that lookahead point: $g(\theta+\mu v)$. The correction is earlier because the gradient sees the anticipated position, not the stale one.

In [ ]:
def grad_quad_w(theta):                         # gradient of L(theta)=0.5*(theta-1)^2.
    return theta - 1.0

theta_n_w = 3.0                                 # current parameter.
v_n_w = -0.4                                    # current velocity already moving left.
mu_n_w = 0.9                                    # momentum memory.
eta_n_w = 0.2                                   # learning rate.
lookahead_w = theta_n_w + mu_n_w * v_n_w        # Nesterov's anticipated parameter.
g_current_w = grad_quad_w(theta_n_w)            # ordinary momentum gradient.
g_look_w = grad_quad_w(lookahead_w)             # Nesterov gradient at lookahead.
print("current theta:", theta_n_w, "lookahead:", round(lookahead_w, 3))
print("gradient current vs lookahead:", round(g_current_w, 3), round(g_look_w, 3))
assert round(lookahead_w, 3) == 2.640

▶ What you'll see: the lookahead point is closer to the optimum, so its gradient is smaller.

In [ ]:
v_mom_next_w = mu_n_w * v_n_w - eta_n_w * g_current_w   # ordinary momentum uses current gradient.
theta_mom_next_w = theta_n_w + v_mom_next_w             # next parameter under ordinary momentum.
v_nes_next_w = mu_n_w * v_n_w - eta_n_w * g_look_w      # Nesterov uses lookahead gradient.
theta_nes_next_w = theta_n_w + v_nes_next_w             # next parameter under Nesterov.
print("ordinary momentum next:", round(theta_mom_next_w, 3))
print("Nesterov next:", round(theta_nes_next_w, 3))
assert round(theta_mom_next_w, 3) == 2.240
assert round(theta_nes_next_w, 3) == 2.312

▶ What you'll see: Nesterov moves less aggressively because it already looked in the direction velocity was carrying it.

In [ ]:
theta_grid_n_w = np.linspace(0.5, 3.2, 160)
loss_grid_n_w = 0.5 * (theta_grid_n_w - 1.0) ** 2
plt.figure(figsize=(4.6, 3))
plt.plot(theta_grid_n_w, loss_grid_n_w, color="navy")
plt.scatter([theta_n_w, lookahead_w, theta_mom_next_w, theta_nes_next_w],
            [0.5*(theta_n_w-1)**2, 0.5*(lookahead_w-1)**2, 0.5*(theta_mom_next_w-1)**2, 0.5*(theta_nes_next_w-1)**2],
            color=["black", "orange", "gray", "teal"], zorder=3)
plt.title("5: Nesterov gradient at the lookahead")
plt.xlabel("theta")
plt.ylabel("loss")
plt.show()

▶ What you'll see: the orange lookahead point sits between the current parameter and the final Nesterov correction.

*Why it's done this way:* momentum can overshoot because it measures slope before applying its stored motion. Nesterov partially fixes that timing mismatch: it estimates where inertia will land, measures the slope there, and uses that slope to correct the velocity before the parameter arrives.

### 6. Scale and memory bookkeeping decide whether the update is usable

The same formula can train well or badly depending on numerical scale. Normalization makes a signal's size interpretable, and memory estimates remind us that optimizer states are real arrays. Momentum stores one velocity per parameter; adaptive optimizers store even more. Here we keep the lesson's normalization and memory arithmetic explicit.

In [ ]:
signal_w = 2.4                              # lesson signal from the scratch pass.
mean_w = 1.0                                # normalization mean.
var_w = 0.25                                # normalization variance.
eps_w = 1e-5                                # numerical stabilizer.
normalized_w = (signal_w - mean_w) / np.sqrt(var_w + eps_w)  # standardize the signal.
print("normalized value:", round(normalized_w, 3))
assert round(normalized_w, 3) == 2.800

▶ What you'll see: the score `2.4` is `2.8` standard deviations above the reference mean.

In [ ]:
vectors_w = 5                               # tiny activation block count.
width_w = 128                               # vector length.
bytes_per_float_w = 4                       # float32 size.
activation_kb_w = vectors_w * width_w * bytes_per_float_w / 1024  # memory in KB.
params_w = 1_000_000                        # example model parameter count.
momentum_mb_w = params_w * bytes_per_float_w / (1024 ** 2)         # one extra velocity buffer.
print("activation memory KB:", round(activation_kb_w, 3))
print("momentum state MB for 1M params:", round(momentum_mb_w, 3))
assert round(activation_kb_w, 3) == 2.500

▶ What you'll see: the tiny block uses `2.5 KB`, while one million momentum velocities already need about `3.8 MB`.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(["activation KB", "velocity MB"], [activation_kb_w, momentum_mb_w], color=["purple", "teal"])
plt.title("6: scale bookkeeping for training")
plt.ylabel("displayed units")
plt.show()

▶ What you'll see: even a simple optimizer has bookkeeping costs beyond the forward activations.

*Why it's done this way:* normalization keeps gradients in a range where $\eta g$ is meaningful, and memory accounting keeps the optimizer honest on real hardware. Momentum is mathematically just a velocity recurrence, but implementation requires storing that velocity at the same shape as the parameters.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, gradients, optimizer recurrences, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for optimizer paths, loss curves, and diagnostic plots.
np.random.seed(0) # make every stochastic example reproducible.

def quadratic_loss(theta, center=0.0, scales=None): # compute a separable quadratic loss for scalar or vector theta.
    theta = np.asarray(theta, dtype=float) # convert input to float array so vector arithmetic is predictable.
    if scales is None: # default to equal curvature in every coordinate.
        scales = np.ones_like(theta) # create unit curvature for each coordinate.
    scales = np.asarray(scales, dtype=float) # convert scales to float array.
    return 0.5 * float(np.sum(scales * (theta - center) ** 2)) # return 1/2 sum_i scale_i (theta_i-center)^2.

def quadratic_grad(theta, center=0.0, scales=None): # compute the gradient of the separable quadratic.
    theta = np.asarray(theta, dtype=float) # convert input to a float array.
    if scales is None: # default to equal curvature.
        scales = np.ones_like(theta) # unit curvature.
    scales = np.asarray(scales, dtype=float) # convert scales to floats.
    return scales * (theta - center) # derivative of 1/2 scale*(theta-center)^2.

def sgd_step(theta, grad, eta): # apply one vanilla SGD step.
    return np.asarray(theta, dtype=float) - eta * np.asarray(grad, dtype=float) # subtract learning-rate-scaled gradient.

def momentum_step(theta, velocity, grad, eta, mu): # apply one classical momentum update.
    velocity = mu * np.asarray(velocity, dtype=float) - eta * np.asarray(grad, dtype=float) # update velocity with memory and current gradient.
    theta = np.asarray(theta, dtype=float) + velocity # move parameters by the new velocity.
    return theta, velocity # return both states because momentum is stateful.

def nesterov_step(theta, velocity, grad_fn, eta, mu): # apply one Nesterov update using a callable gradient.
    lookahead = np.asarray(theta, dtype=float) + mu * np.asarray(velocity, dtype=float) # predict where momentum is carrying theta.
    grad = grad_fn(lookahead) # measure gradient at the lookahead location.
    velocity = mu * velocity - eta * grad # correct the velocity using the lookahead gradient.
    theta = np.asarray(theta, dtype=float) + velocity # move parameters by the corrected velocity.
    return theta, velocity, lookahead, grad # return diagnostic values too.

## 🟢 Basics (warm-up)

### Basic 1 — Compute the lesson's scratch signal

**Goal.** Build the affine-plus-ReLU signal from two inputs, because gradients come from forward computations with shapes and scales. We build it in 2 steps.

In [ ]:
x_b1 = np.array([1.5, -0.5]) # define the two input features from the lesson scratch pass.
w_b1 = np.array([1.4, 0.2]) # define one weight per input feature.
b_b1 = 0.4 # define the scalar bias.
print("x:", x_b1, "w:", w_b1, "b:", b_b1) # inspect the forward-pass ingredients.

In [ ]:
pieces_b1 = x_b1 * w_b1 # compute per-feature contributions to the affine score.
z_b1 = float(np.sum(pieces_b1) + b_b1) # sum contributions and bias.
a_b1 = max(0.0, z_b1) # apply ReLU gating.
print("pieces:", pieces_b1, "z:", round(z_b1, 3), "ReLU:", round(a_b1, 3)) # inspect the shaped signal.
assert round(z_b1, 3) == 2.4 # verify the lesson arithmetic.
plt.figure(figsize=(4, 3)) # create a compact contribution chart.
plt.bar(["w0*x0", "w1*x1", "bias", "ReLU"], [pieces_b1[0], pieces_b1[1], b_b1, a_b1], color="teal") # display each part of the signal.
plt.title("Basic 1: scratch forward signal") # title the plot.
plt.ylabel("value") # label the value scale.
plt.show() # display the plot.

▶ What you'll see: the positive first feature dominates, the second subtracts a little, and ReLU passes `2.4` through.

👀 Takeaway: optimizer updates are driven by gradients that originate in ordinary forward-pass arithmetic.

### Basic 2 — Turn two scores into a softmax probability

**Goal.** Normalize a model score against a baseline, because losses usually compare alternatives rather than trusting raw scores. We build it in 2 steps.

In [ ]:
score_b2 = 2.4 # use the scratch score from the lesson.
baseline_b2 = 0.4 # define a competing baseline score.
scores_b2 = np.array([score_b2, baseline_b2]) # package both scores for softmax.
print("scores:", scores_b2) # inspect the raw comparison values.

In [ ]:
exp_b2 = np.exp(scores_b2) # exponentiate scores into positive weights.
prob_b2 = exp_b2[0] / np.sum(exp_b2) # normalize the first score by total exponential mass.
print("exp scores:", np.round(exp_b2, 3), "probability:", round(float(prob_b2), 3)) # inspect softmax arithmetic.
assert round(float(prob_b2), 3) == 0.881 # verify the lesson probability.
plt.figure(figsize=(4, 3)) # create a compact probability chart.
plt.bar(["target", "baseline"], exp_b2 / exp_b2.sum(), color=["purple", "gray"]) # plot normalized probabilities.
plt.title("Basic 2: softmax comparison") # title the plot.
plt.ylabel("probability") # label the probability axis.
plt.show() # display the plot.

▶ What you'll see: the higher score receives most probability mass but not all of it.

👀 Takeaway: softmax turns score gaps into calibrated comparisons that can produce useful gradients.

### Basic 3 — Apply one vanilla SGD update

**Goal.** Move one scalar parameter with $\theta-\eta g$, because this is the core training knob before momentum adds memory. We build it in 2 steps.

In [ ]:
theta_b3 = 2.0 # define the current scalar parameter.
eta_b3 = 0.07 # define the learning rate.
g_b3 = 1.2 # define the scalar gradient.
print("theta:", theta_b3, "eta:", eta_b3, "gradient:", g_b3) # inspect the update ingredients.

In [ ]:
step_b3 = eta_b3 * g_b3 # compute the amount subtracted from theta.
theta_new_b3 = theta_b3 - step_b3 # apply the vanilla SGD update.
print("step:", round(step_b3, 3), "theta_new:", round(theta_new_b3, 3)) # inspect the new parameter.
assert round(theta_new_b3, 3) == 1.916 # verify the lesson update.
plt.figure(figsize=(4, 3)) # create a before-after chart.
plt.bar(["before", "after"], [theta_b3, theta_new_b3], color=["gray", "teal"]) # compare parameter values.
plt.title("Basic 3: one SGD nudge") # title the plot.
plt.ylabel("theta") # label the parameter axis.
plt.show() # display the plot.

▶ What you'll see: the parameter decreases by exactly `0.084`.

👀 Takeaway: the learning rate scales the gradient into a parameter displacement.

### Basic 4 — Check that a gradient step lowers a quadratic loss

**Goal.** Verify a step on a simple loss bowl, because descent only works when the update direction and step size are sensible. We build it in 3 steps.

In [ ]:
theta_b4 = np.array([2.0]) # start to the right of the optimum.
center_b4 = 0.5 # set the quadratic minimum.
g_b4 = quadratic_grad(theta_b4, center=center_b4) # compute the exact gradient theta-center.
print("gradient:", np.round(g_b4, 3)) # inspect the slope direction.

In [ ]:
eta_b4 = 0.2 # choose a stable learning rate for this one-dimensional bowl.
theta_next_b4 = sgd_step(theta_b4, g_b4, eta_b4) # apply one SGD step.
loss_before_b4 = quadratic_loss(theta_b4, center=center_b4) # compute starting loss.
loss_after_b4 = quadratic_loss(theta_next_b4, center=center_b4) # compute post-step loss.
print("loss before -> after:", round(loss_before_b4, 3), "->", round(loss_after_b4, 3)) # inspect improvement.
assert loss_after_b4 < loss_before_b4 # verify descent.

In [ ]:
grid_b4 = np.linspace(0, 2.2, 100) # create theta values for plotting the bowl.
loss_grid_b4 = 0.5 * (grid_b4 - center_b4) ** 2 # compute quadratic loss values.
plt.figure(figsize=(4, 3)) # create the loss-bowl figure.
plt.plot(grid_b4, loss_grid_b4, color="navy") # draw the loss curve.
plt.scatter([theta_b4[0], theta_next_b4[0]], [loss_before_b4, loss_after_b4], color=["red", "green"]) # mark before and after.
plt.title("Basic 4: descent on a quadratic") # title the plot.
plt.xlabel("theta") # label the parameter axis.
plt.ylabel("loss") # label the loss axis.
plt.show() # display the curve.

▶ What you'll see: the post-step point sits lower on the bowl than the starting point.

👀 Takeaway: a correct gradient direction plus a reasonable learning rate reduces loss locally.

### Basic 5 — See how learning rate changes step length

**Goal.** Compare several learning rates for the same gradient, because $\eta$ decides whether SGD crawls, learns, or overshoots. We build it in 3 steps.

In [ ]:
theta_b5 = 2.0 # start from the same parameter for every learning rate.
g_b5 = 1.2 # hold the gradient fixed.
etas_b5 = np.array([0.01, 0.07, 0.4]) # compare small, lesson-sized, and large steps.
print("learning rates:", etas_b5) # inspect the step sizes.

In [ ]:
nexts_b5 = theta_b5 - etas_b5 * g_b5 # compute one update for every eta.
print("next theta values:", np.round(nexts_b5, 3)) # inspect how far each step moves.
assert round(float(nexts_b5[1]), 3) == 1.916 # verify the lesson-sized update.

In [ ]:
plt.figure(figsize=(4, 3)) # create a step-length comparison.
plt.bar(["0.01", "0.07", "0.40"], theta_b5 - nexts_b5, color="orange") # plot displacement magnitude.
plt.title("Basic 5: learning-rate-scaled steps") # title the plot.
plt.xlabel("eta") # label each learning rate.
plt.ylabel("amount subtracted") # label displacement.
plt.show() # display the bar chart.

▶ What you'll see: the same gradient produces very different parameter movement as `eta` changes.

👀 Takeaway: learning rate is not decoration; it is the conversion factor from slope to motion.

### Basic 6 — Compute one momentum velocity

**Goal.** Add memory to an SGD step, because momentum stores previous motion in a velocity variable. We build it in 2 steps.

In [ ]:
v_prev_b6 = np.array([-0.10, 0.02]) # define previous velocity from earlier steps.
g_b6 = np.array([1.0, -0.5]) # define the current gradient.
mu_b6 = 0.9 # define momentum retention.
eta_b6 = 0.1 # define learning rate.
print("previous velocity:", v_prev_b6, "gradient:", g_b6) # inspect the state before updating.

In [ ]:
v_new_b6 = mu_b6 * v_prev_b6 - eta_b6 * g_b6 # compute the momentum velocity update.
theta_b6 = np.array([2.0, 1.0]) # choose a parameter vector to move.
theta_new_b6 = theta_b6 + v_new_b6 # move by velocity rather than by raw gradient only.
print("new velocity:", np.round(v_new_b6, 3), "new theta:", np.round(theta_new_b6, 3)) # inspect updated state.
assert np.allclose(np.round(v_new_b6, 3), [-0.19, 0.068]) # verify the velocity arithmetic.
plt.figure(figsize=(4, 3)) # create a velocity component chart.
plt.bar(["v0", "v1"], v_new_b6, color="teal") # show each velocity coordinate.
plt.axhline(0, color="black", linewidth=0.8) # separate positive and negative motion.
plt.title("Basic 6: momentum velocity") # title the plot.
plt.ylabel("velocity") # label velocity scale.
plt.show() # display the plot.

▶ What you'll see: the new velocity combines old motion and the current downhill push.

👀 Takeaway: momentum makes the optimizer stateful by carrying velocity between updates.

### Basic 7 — Accumulate velocity under repeated gradients

**Goal.** Watch consistent gradients build speed, because momentum rewards repeated evidence. We build it in 3 steps.

In [ ]:
grads_b7 = np.ones(5) * 1.0 # create five identical positive gradients.
eta_b7 = 0.1 # choose a simple learning rate.
mu_b7 = 0.9 # choose a common momentum value.
print("gradients:", grads_b7) # inspect repeated evidence.

In [ ]:
v_b7 = 0.0 # start from rest.
vels_b7 = [] # store the scalar velocity after each step.
for grad_b7 in grads_b7: # apply identical gradients repeatedly.
    v_b7 = mu_b7 * v_b7 - eta_b7 * grad_b7 # update velocity.
    vels_b7.append(v_b7) # record velocity.
print("velocities:", np.round(vels_b7, 3)) # inspect acceleration.
assert round(vels_b7[-1], 3) == -0.410 # verify accumulated velocity.

In [ ]:
plt.figure(figsize=(4, 3)) # create a velocity curve.
plt.plot(range(1, 6), vels_b7, marker="o", color="purple") # plot velocity over repeated gradients.
plt.title("Basic 7: repeated gradients build velocity") # title the plot.
plt.xlabel("step") # label the step axis.
plt.ylabel("velocity") # label velocity.
plt.show() # display the curve.

▶ What you'll see: velocity grows more negative each step, so the parameter would move faster downhill.

👀 Takeaway: momentum accelerates when gradients keep agreeing.

### Basic 8 — Let alternating gradients cancel in velocity

**Goal.** Show the smoothing effect of momentum, because noisy coordinates often flip sign across minibatches. We build it in 3 steps.

In [ ]:
grads_b8 = np.array([1.0, -1.0, 1.0, -1.0]) # define alternating scalar gradients.
mu_b8 = 0.9 # retain most velocity.
eta_b8 = 0.1 # scale each gradient contribution.
print("alternating gradients:", grads_b8) # inspect the noisy sequence.

In [ ]:
v_b8 = 0.0 # start with no velocity.
vels_b8 = [] # record velocity after each alternating gradient.
for grad_b8 in grads_b8: # step through the noisy gradient sequence.
    v_b8 = mu_b8 * v_b8 - eta_b8 * grad_b8 # update velocity with memory.
    vels_b8.append(v_b8) # store the velocity.
print("velocities:", np.round(vels_b8, 3)) # inspect partial cancellation.
assert abs(vels_b8[-1]) < 0.04 # verify cancellation keeps velocity small.

In [ ]:
plt.figure(figsize=(4, 3)) # create a cancellation plot.
plt.plot(range(1, 5), vels_b8, marker="o", color="crimson") # plot alternating-gradient velocities.
plt.axhline(0, color="black", linewidth=0.8) # show zero velocity reference.
plt.title("Basic 8: alternating gradients cancel") # title the plot.
plt.xlabel("step") # label steps.
plt.ylabel("velocity") # label velocity.
plt.show() # display the curve.

▶ What you'll see: velocity oscillates near zero instead of growing without bound.

👀 Takeaway: momentum damps directions where minibatch gradients keep disagreeing.

### Basic 9 — Compute a Nesterov lookahead point

**Goal.** Find $\theta+\mu v$, because Nesterov measures the gradient where momentum is about to carry the parameter. We build it in 2 steps.

In [ ]:
theta_b9 = np.array([3.0]) # define the current parameter.
velocity_b9 = np.array([-0.4]) # define current leftward velocity.
mu_b9 = 0.9 # define momentum retention.
print("theta:", theta_b9, "velocity:", velocity_b9) # inspect the lookahead ingredients.

In [ ]:
lookahead_b9 = theta_b9 + mu_b9 * velocity_b9 # compute anticipated parameter location.
g_look_b9 = quadratic_grad(lookahead_b9, center=1.0) # compute gradient at the lookahead on a simple quadratic.
print("lookahead:", np.round(lookahead_b9, 3), "lookahead gradient:", np.round(g_look_b9, 3)) # inspect Nesterov's gradient location.
assert round(float(lookahead_b9[0]), 3) == 2.64 # verify the lesson lookahead value.
plt.figure(figsize=(4, 3)) # create a simple location plot.
plt.scatter([theta_b9[0], lookahead_b9[0], 1.0], [0, 0, 0], color=["black", "orange", "green"], s=80) # show current, lookahead, and optimum.
plt.yticks([]) # remove unused y-axis ticks.
plt.title("Basic 9: Nesterov lookahead") # title the plot.
plt.xlabel("theta") # label the parameter axis.
plt.show() # display the plot.

▶ What you'll see: the lookahead point lies between the current parameter and the optimum because velocity is already moving left.

👀 Takeaway: Nesterov changes where the gradient is measured, not just how it is scaled.

### Basic 10 — Normalize a signal and estimate velocity memory

**Goal.** Do scale and memory bookkeeping, because stable optimization depends on normalized signals and extra optimizer state. We build it in 3 steps.

In [ ]:
signal_b10 = 2.4 # define the lesson signal value.
mean_b10 = 1.0 # define normalization mean.
var_b10 = 0.25 # define normalization variance.
eps_b10 = 1e-5 # define a small stabilizer.
print("signal, mean, variance:", signal_b10, mean_b10, var_b10) # inspect normalization inputs.

In [ ]:
normed_b10 = (signal_b10 - mean_b10) / np.sqrt(var_b10 + eps_b10) # standardize the signal.
activation_kb_b10 = 5 * 128 * 4 / 1024 # compute memory for five float32 vectors of length 128.
print("normalized signal:", round(normed_b10, 3), "activation KB:", round(activation_kb_b10, 3)) # inspect scale and memory.
assert round(normed_b10, 3) == 2.8 # verify the lesson normalization.
assert round(activation_kb_b10, 3) == 2.5 # verify the lesson memory number.

In [ ]:
params_b10 = 1_000_000 # choose one million parameters for a concrete optimizer-state estimate.
velocity_mb_b10 = params_b10 * 4 / (1024 ** 2) # one float32 velocity per parameter.
plt.figure(figsize=(4, 3)) # create a memory comparison chart.
plt.bar(["activation KB", "velocity MB"], [activation_kb_b10, velocity_mb_b10], color=["purple", "teal"]) # compare two bookkeeping quantities.
plt.title("Basic 10: scale and optimizer state") # title the plot.
plt.ylabel("displayed units") # label units as displayed.
plt.show() # display the chart.

▶ What you'll see: a single normalized value and a concrete reminder that momentum stores an extra array.

👀 Takeaway: optimizer formulas are only usable when signal scale and memory state are tracked carefully.

## 🟡 Easy

### Easy 1 — Compare plain SGD and momentum on noisy gradients

**Goal.** Run both optimizers on the same gradient sequence, because momentum should smooth zig-zag while preserving consistent movement. We build it in 4 steps.

In [ ]:
grads_e1 = np.array([[1.0, 0.7], [1.0, -0.6], [1.0, 0.5], [1.0, -0.4], [1.0, 0.3]]) # steady x-gradient with alternating y-noise.
eta_e1 = 0.1 # choose learning rate.
mu_e1 = 0.9 # choose momentum coefficient.
start_e1 = np.array([2.0, 1.0]) # set identical starting parameters.
print("gradient sequence shape:", grads_e1.shape) # inspect the sequence length and dimension.

In [ ]:
path_sgd_e1 = [start_e1.copy()] # store plain SGD path.
for grad_e1 in grads_e1: # apply vanilla SGD to every gradient.
    path_sgd_e1.append(path_sgd_e1[-1] - eta_e1 * grad_e1) # subtract scaled gradient.
path_sgd_e1 = np.array(path_sgd_e1) # convert to array for plotting.
print("plain SGD final:", np.round(path_sgd_e1[-1], 3)) # inspect final position.

In [ ]:
path_mom_e1 = [start_e1.copy()] # store momentum path.
v_e1 = np.zeros(2) # start momentum from rest.
for grad_e1 in grads_e1: # apply momentum to every gradient.
    new_theta_e1, v_e1 = momentum_step(path_mom_e1[-1], v_e1, grad_e1, eta_e1, mu_e1) # update parameter and velocity.
    path_mom_e1.append(new_theta_e1) # store new position.
path_mom_e1 = np.array(path_mom_e1) # convert to array.
print("momentum final:", np.round(path_mom_e1[-1], 3), "final velocity:", np.round(v_e1, 3)) # inspect final state.
assert path_mom_e1[-1, 0] < path_sgd_e1[-1, 0] # verify momentum moved farther in the consistent direction.

In [ ]:
plt.figure(figsize=(5, 3.5)) # create path comparison.
plt.plot(path_sgd_e1[:, 0], path_sgd_e1[:, 1], marker="o", label="SGD", color="gray") # draw plain path.
plt.plot(path_mom_e1[:, 0], path_mom_e1[:, 1], marker="o", label="Momentum", color="teal") # draw momentum path.
plt.title("Easy 1: momentum vs noisy SGD") # title the plot.
plt.xlabel("theta[0]") # label first coordinate.
plt.ylabel("theta[1]") # label second coordinate.
plt.legend() # show method labels.
plt.show() # display comparison.

▶ What you'll see: momentum travels farther left while reducing the visible up-down jitter.

👀 Takeaway: momentum is useful when one direction is consistently helpful and another is noisy.

### Easy 2 — Simulate Nesterov on a quadratic bowl

**Goal.** Compare ordinary momentum with Nesterov, because lookahead gradients can correct inertia before it overshoots. We build it in 4 steps.

In [ ]:
eta_e2 = 0.18 # choose a learning rate large enough to show momentum behavior.
mu_e2 = 0.85 # choose momentum retention.
steps_e2 = 18 # choose a short trajectory.
center_e2 = 1.0 # set the optimum of the scalar quadratic.
print("eta, mu, steps:", eta_e2, mu_e2, steps_e2) # inspect simulation settings.

In [ ]:
theta_m_e2 = np.array([3.0]) # initialize ordinary momentum parameter.
v_m_e2 = np.array([0.0]) # initialize ordinary momentum velocity.
path_m_e2 = [theta_m_e2.copy()] # store ordinary momentum path.
for _ in range(steps_e2): # iterate momentum updates.
    grad_e2 = quadratic_grad(theta_m_e2, center=center_e2) # gradient at current theta.
    theta_m_e2, v_m_e2 = momentum_step(theta_m_e2, v_m_e2, grad_e2, eta_e2, mu_e2) # ordinary momentum update.
    path_m_e2.append(theta_m_e2.copy()) # store path point.
path_m_e2 = np.array(path_m_e2).ravel() # flatten scalar path.
print("ordinary final:", round(float(path_m_e2[-1]), 3)) # inspect final parameter.

In [ ]:
theta_n_e2 = np.array([3.0]) # initialize Nesterov parameter.
v_n_e2 = np.array([0.0]) # initialize Nesterov velocity.
path_n_e2 = [theta_n_e2.copy()] # store Nesterov path.
for _ in range(steps_e2): # iterate Nesterov updates.
    grad_fn_e2 = lambda th: quadratic_grad(th, center=center_e2) # define gradient function for lookahead.
    theta_n_e2, v_n_e2, look_e2, grad_look_e2 = nesterov_step(theta_n_e2, v_n_e2, grad_fn_e2, eta_e2, mu_e2) # Nesterov update.
    path_n_e2.append(theta_n_e2.copy()) # store path point.
path_n_e2 = np.array(path_n_e2).ravel() # flatten scalar path.
print("Nesterov final:", round(float(path_n_e2[-1]), 3)) # inspect final parameter.
assert abs(path_n_e2[-1] - center_e2) < abs(path_m_e2[-1] - center_e2) # verify Nesterov ends closer here.

In [ ]:
plt.figure(figsize=(5, 3)) # create convergence plot.
plt.plot(np.abs(path_m_e2 - center_e2), marker="o", label="momentum", color="gray") # plot distance to optimum.
plt.plot(np.abs(path_n_e2 - center_e2), marker="o", label="Nesterov", color="teal") # plot Nesterov distance.
plt.yscale("log") # use log scale to see late differences.
plt.title("Easy 2: distance to optimum") # title the convergence plot.
plt.xlabel("step") # label step axis.
plt.ylabel("|theta - optimum|, log") # label distance axis.
plt.legend() # show method labels.
plt.show() # display plot.

▶ What you'll see: both methods accelerate, and in this setting Nesterov finishes closer to the optimum.

👀 Takeaway: Nesterov's lookahead gradient can make momentum's correction better timed.

### Easy 3 — Train a one-parameter linear model with SGD

**Goal.** Fit a slope from data using minibatch SGD, because optimizer updates ultimately move model parameters to reduce prediction error. We build it in 4 steps.

In [ ]:
x_e3 = np.linspace(-1, 1, 9) # create one-dimensional training inputs.
y_e3 = 2.0 * x_e3 + 0.3 # create exact targets from a true slope and bias.
w_e3 = 0.0 # initialize slope at zero.
bias_e3 = 0.0 # initialize bias at zero.
print("first targets:", np.round(y_e3[:3], 3)) # inspect the supervised signal.

In [ ]:
eta_e3 = 0.15 # choose a stable learning rate.
losses_e3 = [] # store mean squared error over epochs.
for epoch_e3 in range(30): # make repeated passes over the tiny dataset.
    for xi_e3, yi_e3 in zip(x_e3, y_e3): # use one example at a time: true SGD.
        pred_e3 = w_e3 * xi_e3 + bias_e3 # compute current prediction.
        err_e3 = pred_e3 - yi_e3 # derivative of 1/2 error^2 with respect to prediction.
        w_e3 -= eta_e3 * err_e3 * xi_e3 # update slope using chain rule.
        bias_e3 -= eta_e3 * err_e3 # update bias.
    losses_e3.append(float(np.mean((w_e3 * x_e3 + bias_e3 - y_e3) ** 2))) # record full-data MSE.
print("learned w,b:", round(w_e3, 3), round(bias_e3, 3)) # inspect learned parameters.
assert losses_e3[-1] < losses_e3[0] # verify training improved.

In [ ]:
preds_e3 = w_e3 * x_e3 + bias_e3 # compute final predictions.
print("final MSE:", round(losses_e3[-1], 6)) # inspect final fit quality.
assert abs(w_e3 - 2.0) < 0.05 # verify slope recovery.

In [ ]:
plt.figure(figsize=(5, 3)) # create data-fit plot.
plt.scatter(x_e3, y_e3, label="data", color="black") # plot training data.
plt.plot(x_e3, preds_e3, label="SGD fit", color="teal") # plot learned line.
plt.title("Easy 3: SGD learns a line") # title the plot.
plt.xlabel("x") # label input axis.
plt.ylabel("y") # label target axis.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: the fitted line nearly overlaps the exact data-generating line.

👀 Takeaway: the same update rule that moves toy quadratics also trains model parameters through gradients.

### Easy 4 — Compare normalized and unnormalized gradients

**Goal.** Show how feature scale changes gradient size, because ignoring scale can make a sound update train badly. We build it in 4 steps.

In [ ]:
x_raw_e4 = np.array([1.0, 10.0, 20.0, 30.0]) # create one feature with a large numeric scale.
y_e4 = 0.5 * x_raw_e4 # create simple targets.
w_raw_e4 = 0.0 # initialize one slope for raw inputs.
w_norm_e4 = 0.0 # initialize one slope for normalized inputs.
x_norm_e4 = (x_raw_e4 - x_raw_e4.mean()) / x_raw_e4.std() # standardize the feature.
print("raw std:", round(float(x_raw_e4.std()), 3), "normalized std:", round(float(x_norm_e4.std()), 3)) # inspect scale.

In [ ]:
pred_raw_e4 = w_raw_e4 * x_raw_e4 # compute raw predictions at initialization.
grad_raw_e4 = float(np.mean((pred_raw_e4 - y_e4) * x_raw_e4)) # gradient for raw slope.
pred_norm_e4 = w_norm_e4 * x_norm_e4 # compute normalized predictions at initialization.
grad_norm_e4 = float(np.mean((pred_norm_e4 - y_e4) * x_norm_e4)) # gradient for normalized slope.
print("raw gradient:", round(grad_raw_e4, 3), "normalized gradient:", round(grad_norm_e4, 3)) # compare gradient magnitudes.
assert abs(grad_raw_e4) > abs(grad_norm_e4) # verify scale changed gradient size.

In [ ]:
eta_e4 = 0.01 # choose the same learning rate for both.
w_raw_next_e4 = w_raw_e4 - eta_e4 * grad_raw_e4 # update raw slope.
w_norm_next_e4 = w_norm_e4 - eta_e4 * grad_norm_e4 # update normalized slope.
print("one-step raw w:", round(w_raw_next_e4, 3), "normalized w:", round(w_norm_next_e4, 3)) # inspect step sizes.

In [ ]:
plt.figure(figsize=(4, 3)) # create gradient comparison chart.
plt.bar(["raw grad", "normalized grad"], [abs(grad_raw_e4), abs(grad_norm_e4)], color=["red", "teal"]) # compare magnitudes.
plt.title("Easy 4: scale changes gradient size") # title the plot.
plt.ylabel("absolute gradient") # label magnitude.
plt.show() # display plot.

▶ What you'll see: the raw feature creates a much larger slope gradient under the same model error.

👀 Takeaway: normalization makes a single learning rate more meaningful across coordinates.

### Easy 5 — Tune momentum coefficient on a valley

**Goal.** Sweep $\mu$ values, because too little memory is slow while too much can overshoot. We build it in 4 steps.

In [ ]:
mus_e5 = np.array([0.0, 0.5, 0.9]) # compare no momentum, moderate momentum, and high momentum.
eta_e5 = 0.12 # choose a shared learning rate.
scales_e5 = np.array([1.0, 0.08]) # define a shallow second direction to create a valley.
start_e5 = np.array([3.0, 2.0]) # start away from the optimum.
print("momentum coefficients:", mus_e5) # inspect sweep values.

In [ ]:
final_losses_e5 = [] # store final loss for each momentum coefficient.
paths_e5 = [] # store trajectories for plotting.
for mu_e5 in mus_e5: # run one optimizer per momentum value.
    theta_e5 = start_e5.copy() # reset parameter.
    v_e5 = np.zeros(2) # reset velocity.
    path_e5 = [theta_e5.copy()] # store path.
    for _ in range(35): # run a short optimization.
        grad_e5 = quadratic_grad(theta_e5, center=0.0, scales=scales_e5) # compute valley gradient.
        theta_e5, v_e5 = momentum_step(theta_e5, v_e5, grad_e5, eta_e5, mu_e5) # apply momentum update.
        path_e5.append(theta_e5.copy()) # store current parameter.
    final_losses_e5.append(quadratic_loss(theta_e5, center=0.0, scales=scales_e5)) # store final loss.
    paths_e5.append(np.array(path_e5)) # store path array.
print("final losses:", np.round(final_losses_e5, 4)) # inspect sweep outcome.
assert min(final_losses_e5) < final_losses_e5[0] # verify some momentum helps here.

In [ ]:
best_idx_e5 = int(np.argmin(final_losses_e5)) # find best final loss.
best_mu_e5 = mus_e5[best_idx_e5] # read best momentum coefficient.
print("best mu in this toy sweep:", best_mu_e5) # inspect selected coefficient.

In [ ]:
plt.figure(figsize=(5, 3.5)) # create path sweep plot.
for path_e5, mu_e5 in zip(paths_e5, mus_e5): # draw each momentum trajectory.
    plt.plot(path_e5[:, 0], path_e5[:, 1], marker="o", markersize=3, label=f"mu={mu_e5}") # plot path.
plt.title("Easy 5: momentum coefficient sweep") # title the plot.
plt.xlabel("theta[0]") # label coordinate 0.
plt.ylabel("theta[1]") # label coordinate 1.
plt.legend() # show mu labels.
plt.show() # display paths.

▶ What you'll see: different momentum values create different speeds and path shapes through the same valley.

👀 Takeaway: $\mu$ is a stability-speed knob that should be chosen for the training dynamics, not set blindly.

## 🔴 Advanced

### Advanced 1 — Trace SGD, Momentum, and Nesterov on the same ill-conditioned valley

**Goal.** Compare full trajectories on a two-dimensional valley, because optimizer differences matter most when curvature and scale differ by coordinate. We build it in 5 steps.

In [ ]:
scales_a1 = np.array([1.0, 0.04]) # create steep and shallow coordinates.
start_a1 = np.array([3.0, 2.5]) # start away from the optimum.
eta_a1 = 0.16 # choose a shared learning rate.
mu_a1 = 0.85 # choose a shared momentum coefficient.
steps_a1 = 45 # choose enough steps to show trajectory differences.
print("scales:", scales_a1, "eta:", eta_a1, "mu:", mu_a1) # inspect setup.

In [ ]:
path_sgd_a1 = [start_a1.copy()] # store vanilla SGD path.
theta_sgd_a1 = start_a1.copy() # initialize SGD parameter.
for _ in range(steps_a1): # run vanilla SGD.
    grad_a1 = quadratic_grad(theta_sgd_a1, center=0.0, scales=scales_a1) # compute gradient.
    theta_sgd_a1 = sgd_step(theta_sgd_a1, grad_a1, eta_a1) # apply SGD.
    path_sgd_a1.append(theta_sgd_a1.copy()) # store path.
path_sgd_a1 = np.array(path_sgd_a1) # convert to array.
print("SGD final loss:", round(quadratic_loss(path_sgd_a1[-1], scales=scales_a1), 4)) # inspect final loss.

In [ ]:
path_mom_a1 = [start_a1.copy()] # store momentum path.
theta_mom_a1 = start_a1.copy() # initialize momentum parameter.
v_mom_a1 = np.zeros(2) # initialize velocity.
for _ in range(steps_a1): # run classical momentum.
    grad_a1 = quadratic_grad(theta_mom_a1, center=0.0, scales=scales_a1) # gradient at current point.
    theta_mom_a1, v_mom_a1 = momentum_step(theta_mom_a1, v_mom_a1, grad_a1, eta_a1, mu_a1) # update momentum.
    path_mom_a1.append(theta_mom_a1.copy()) # store path.
path_mom_a1 = np.array(path_mom_a1) # convert to array.
print("Momentum final loss:", round(quadratic_loss(path_mom_a1[-1], scales=scales_a1), 4)) # inspect final loss.

In [ ]:
path_nes_a1 = [start_a1.copy()] # store Nesterov path.
theta_nes_a1 = start_a1.copy() # initialize Nesterov parameter.
v_nes_a1 = np.zeros(2) # initialize Nesterov velocity.
for _ in range(steps_a1): # run Nesterov momentum.
    grad_fn_a1 = lambda th: quadratic_grad(th, center=0.0, scales=scales_a1) # gradient function for lookahead.
    theta_nes_a1, v_nes_a1, look_a1, grad_look_a1 = nesterov_step(theta_nes_a1, v_nes_a1, grad_fn_a1, eta_a1, mu_a1) # Nesterov update.
    path_nes_a1.append(theta_nes_a1.copy()) # store path.
path_nes_a1 = np.array(path_nes_a1) # convert to array.
losses_final_a1 = [quadratic_loss(path_sgd_a1[-1], scales=scales_a1), quadratic_loss(path_mom_a1[-1], scales=scales_a1), quadratic_loss(path_nes_a1[-1], scales=scales_a1)] # collect losses.
print("final losses:", np.round(losses_final_a1, 4)) # inspect all methods.
assert min(losses_final_a1) < losses_final_a1[0] # verify a momentum method beats plain SGD here.

In [ ]:
plt.figure(figsize=(5, 4)) # create trajectory comparison plot.
plt.plot(path_sgd_a1[:, 0], path_sgd_a1[:, 1], label="SGD", color="gray") # draw SGD path.
plt.plot(path_mom_a1[:, 0], path_mom_a1[:, 1], label="Momentum", color="orange") # draw momentum path.
plt.plot(path_nes_a1[:, 0], path_nes_a1[:, 1], label="Nesterov", color="teal") # draw Nesterov path.
plt.scatter([0], [0], color="black", marker="x", label="optimum") # mark the minimum.
plt.title("Advanced 1: optimizer paths in a valley") # title the plot.
plt.xlabel("theta[0]") # label steep coordinate.
plt.ylabel("theta[1]") # label shallow coordinate.
plt.legend() # show method labels.
plt.show() # display plot.

▶ What you'll see: momentum-based paths move through the shallow valley differently from plain SGD.

👀 Takeaway: optimizer choice is most visible when curvature, scale, and memory interact.

### Advanced 2 — Measure minibatch noise and momentum smoothing statistically

**Goal.** Quantify how velocity reduces gradient variance, because momentum behaves like a low-pass filter over noisy minibatch gradients. We build it in 4 steps.

In [ ]:
rng_a2 = np.random.default_rng(2) # create reproducible noise.
true_grad_a2 = np.array([0.8, 0.0]) # define a persistent gradient in coordinate 0 only.
noise_a2 = rng_a2.normal(scale=[0.15, 0.8], size=(80, 2)) # create much larger noise in coordinate 1.
grads_a2 = true_grad_a2 + noise_a2 # build noisy minibatch gradient estimates.
print("gradient std:", np.round(grads_a2.std(axis=0), 3)) # inspect raw gradient variability.

In [ ]:
eta_a2 = 0.05 # choose learning rate for velocity updates.
mu_a2 = 0.9 # choose momentum retention.
v_a2 = np.zeros(2) # initialize velocity.
vels_a2 = [] # record velocity at every minibatch.
for grad_a2 in grads_a2: # feed noisy gradients through momentum.
    v_a2 = mu_a2 * v_a2 - eta_a2 * grad_a2 # update velocity.
    vels_a2.append(v_a2.copy()) # store smoothed state.
vels_a2 = np.array(vels_a2) # convert to array.
print("velocity std after warmup:", np.round(vels_a2[20:].std(axis=0), 3)) # inspect smoothed variability.
assert vels_a2[20:, 1].std() < eta_a2 * grads_a2[:, 1].std() * 3 # verify velocity is bounded despite noisy y.

In [ ]:
mean_grad_a2 = grads_a2.mean(axis=0) # estimate average gradient.
mean_vel_a2 = vels_a2[20:].mean(axis=0) # estimate average late velocity.
print("mean gradient:", np.round(mean_grad_a2, 3), "mean late velocity:", np.round(mean_vel_a2, 3)) # inspect persistent direction.

In [ ]:
plt.figure(figsize=(5, 3)) # create smoothing plot.
plt.plot(grads_a2[:, 1], alpha=0.45, label="raw noisy y-gradient", color="red") # draw noisy coordinate.
plt.plot(vels_a2[:, 1] / (-eta_a2), label="velocity-implied average", color="teal") # scale velocity for comparison.
plt.title("Advanced 2: momentum filters noise") # title the plot.
plt.xlabel("minibatch step") # label step axis.
plt.ylabel("coordinate 1 signal") # label signal axis.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: raw minibatch gradients jump around, while the velocity-derived curve changes more smoothly.

👀 Takeaway: momentum uses memory to average noisy gradients while still following persistent directions.

### Advanced 3 — Show overshoot from too large a learning rate

**Goal.** Compare stable and unstable step sizes on the same loss, because a correct gradient can still fail when $\eta$ is too large. We build it in 4 steps.

In [ ]:
etas_a3 = [0.4, 2.2] # choose one stable and one too-large learning rate for L=0.5*theta^2.
steps_a3 = 12 # simulate a short run.
start_a3 = np.array([1.5]) # start away from the minimum.
print("learning rates:", etas_a3) # inspect comparison values.

In [ ]:
paths_a3 = [] # store scalar paths for each learning rate.
for eta_a3 in etas_a3: # run one SGD sequence per eta.
    theta_a3 = start_a3.copy() # reset parameter.
    path_a3 = [float(theta_a3[0])] # store initial value.
    for _ in range(steps_a3): # apply repeated gradient steps.
        grad_a3 = quadratic_grad(theta_a3, center=0.0) # gradient is theta.
        theta_a3 = sgd_step(theta_a3, grad_a3, eta_a3) # apply SGD.
        path_a3.append(float(theta_a3[0])) # store scalar theta.
    paths_a3.append(np.array(path_a3)) # save path.
print("final theta values:", [round(float(p[-1]), 3) for p in paths_a3]) # inspect stability.
assert abs(paths_a3[0][-1]) < abs(start_a3[0]) # stable eta improves.
assert abs(paths_a3[1][-1]) > abs(start_a3[0]) # too-large eta diverges in magnitude.

In [ ]:
loss_paths_a3 = [0.5 * p ** 2 for p in paths_a3] # compute loss for every path point.
print("final losses:", [round(float(lp[-1]), 3) for lp in loss_paths_a3]) # inspect loss outcome.

In [ ]:
plt.figure(figsize=(5, 3)) # create stability comparison.
plt.plot(loss_paths_a3[0], marker="o", label="eta=0.4 stable", color="teal") # draw stable loss.
plt.plot(loss_paths_a3[1], marker="o", label="eta=2.2 unstable", color="red") # draw unstable loss.
plt.yscale("log") # log scale shows divergence clearly.
plt.title("Advanced 3: step size can destabilize SGD") # title the plot.
plt.xlabel("step") # label update number.
plt.ylabel("loss, log scale") # label loss.
plt.legend() # show eta labels.
plt.show() # display plot.

▶ What you'll see: the stable learning rate decays loss, while the too-large one bounces across the minimum with growing magnitude.

👀 Takeaway: gradient formulas do not guarantee learning unless the step size respects the loss scale.

### Advanced 4 — Train logistic regression with Nesterov from scratch

**Goal.** Optimize a small classifier with Nesterov momentum, because the same lookahead rule works for real differentiable losses, not just quadratics. We build it in 5 steps.

In [ ]:
X_a4 = np.array([[-2.0, -1.0], [-1.5, -0.7], [1.0, 1.2], [1.8, 1.0], [2.2, 1.6], [-2.2, -1.3]]) # tiny separable dataset.
y_a4 = np.array([0, 0, 1, 1, 1, 0], dtype=float) # binary labels.
w_a4 = np.zeros(2) # initialize logistic weights.
b_a4 = 0.0 # initialize logistic bias.
print("dataset shape:", X_a4.shape) # inspect examples and features.

In [ ]:
def sigmoid_a4(z): # define logistic sigmoid using NumPy only.
    return 1.0 / (1.0 + np.exp(-z)) # map real scores to probabilities.

def loss_grad_a4(w, b): # compute binary cross-entropy loss and gradients.
    logits_a4 = X_a4 @ w + b # compute linear scores.
    probs_a4 = sigmoid_a4(logits_a4) # convert scores to probabilities.
    eps_a4 = 1e-9 # avoid log(0) in the diagnostic loss.
    loss_a4 = -float(np.mean(y_a4 * np.log(probs_a4 + eps_a4) + (1 - y_a4) * np.log(1 - probs_a4 + eps_a4))) # mean cross-entropy.
    err_a4 = probs_a4 - y_a4 # derivative of loss with respect to logits.
    grad_w_a4 = X_a4.T @ err_a4 / len(y_a4) # gradient with respect to weights.
    grad_b_a4 = float(np.mean(err_a4)) # gradient with respect to bias.
    return loss_a4, grad_w_a4, grad_b_a4 # return diagnostics and gradients.

loss0_a4, gw0_a4, gb0_a4 = loss_grad_a4(w_a4, b_a4) # compute initial loss and gradient.
print("initial loss:", round(loss0_a4, 3), "grad_w:", np.round(gw0_a4, 3), "grad_b:", round(gb0_a4, 3)) # inspect initial state.
assert round(loss0_a4, 3) == 0.693 # verify untrained logistic loss.

In [ ]:
eta_a4 = 0.6 # choose a learning rate for the tiny normalized-ish data.
mu_a4 = 0.8 # choose Nesterov momentum coefficient.
vw_a4 = np.zeros_like(w_a4) # initialize weight velocity.
vb_a4 = 0.0 # initialize bias velocity.
losses_a4 = [] # store loss curve.
for _ in range(60): # run Nesterov updates.
    w_look_a4 = w_a4 + mu_a4 * vw_a4 # look ahead for weights.
    b_look_a4 = b_a4 + mu_a4 * vb_a4 # look ahead for bias.
    loss_a4, gw_a4, gb_a4 = loss_grad_a4(w_look_a4, b_look_a4) # measure gradient at lookahead.
    vw_a4 = mu_a4 * vw_a4 - eta_a4 * gw_a4 # update weight velocity.
    vb_a4 = mu_a4 * vb_a4 - eta_a4 * gb_a4 # update bias velocity.
    w_a4 = w_a4 + vw_a4 # move weights.
    b_a4 = b_a4 + vb_a4 # move bias.
    losses_a4.append(loss_grad_a4(w_a4, b_a4)[0]) # record actual current loss.
print("final loss:", round(losses_a4[-1], 4), "weights:", np.round(w_a4, 3), "bias:", round(b_a4, 3)) # inspect fit.
assert losses_a4[-1] < losses_a4[0] # verify training improved.

In [ ]:
probs_a4 = sigmoid_a4(X_a4 @ w_a4 + b_a4) # compute final class probabilities.
preds_a4 = (probs_a4 >= 0.5).astype(int) # threshold probabilities.
acc_a4 = float(np.mean(preds_a4 == y_a4)) # compute training accuracy.
print("probabilities:", np.round(probs_a4, 3), "accuracy:", acc_a4) # inspect classifier quality.
assert acc_a4 == 1.0 # verify perfect separation on this tiny dataset.

In [ ]:
plt.figure(figsize=(5, 3)) # create logistic training plot.
plt.plot(losses_a4, color="teal") # draw loss curve.
plt.title("Advanced 4: Nesterov logistic training") # title the plot.
plt.xlabel("step") # label update step.
plt.ylabel("cross-entropy loss") # label loss.
plt.show() # display plot.

▶ What you'll see: cross-entropy drops quickly and the final probabilities separate the two classes.

👀 Takeaway: Nesterov is just a lookahead version of gradient descent, so it applies wherever you can compute gradients.

### Advanced 5 — Estimate optimizer state memory at model scale

**Goal.** Compare parameter, gradient, and momentum memory, because optimizer design must fit hardware as well as math. We build it in 4 steps.

In [ ]:
param_counts_a5 = np.array([1_000_000, 10_000_000, 100_000_000]) # choose model sizes.
bytes_float_a5 = 4 # float32 bytes per scalar.
print("parameter counts:", param_counts_a5) # inspect sizes.

In [ ]:
param_mb_a5 = param_counts_a5 * bytes_float_a5 / (1024 ** 2) # memory for parameters alone.
grad_mb_a5 = param_mb_a5 # gradients usually need the same shape as parameters.
velocity_mb_a5 = param_mb_a5 # momentum velocity also has the same shape.
total_momentum_mb_a5 = param_mb_a5 + grad_mb_a5 + velocity_mb_a5 # rough training memory for these three arrays.
print("param MB:", np.round(param_mb_a5, 1)) # inspect parameter memory.
print("param+grad+velocity MB:", np.round(total_momentum_mb_a5, 1)) # inspect momentum training state.
assert round(float(param_mb_a5[0]), 1) == 3.8 # verify 1M float32 parameters are about 3.8 MB.

In [ ]:
overhead_factor_a5 = total_momentum_mb_a5 / param_mb_a5 # compute relative overhead.
print("training-state multiplier:", overhead_factor_a5) # inspect memory multiplier.
assert np.allclose(overhead_factor_a5, 3.0) # verify params+grads+velocity is 3x parameter memory.

In [ ]:
plt.figure(figsize=(5, 3)) # create memory scaling chart.
plt.plot(param_counts_a5 / 1e6, param_mb_a5, marker="o", label="parameters", color="gray") # plot parameter memory.
plt.plot(param_counts_a5 / 1e6, total_momentum_mb_a5, marker="o", label="params+grads+velocity", color="teal") # plot training state memory.
plt.title("Advanced 5: momentum state scales with model size") # title the plot.
plt.xlabel("parameters (millions)") # label x-axis.
plt.ylabel("memory (MB)") # label y-axis.
plt.legend() # show memory categories.
plt.show() # display chart.

▶ What you'll see: memory grows linearly with parameter count, and momentum training state is roughly triple parameter memory before activations.

👀 Takeaway: momentum's velocity is an extra full-size model state, so optimizer choice has hardware consequences.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Momentum turns noisy gradient steps into motion with memory; Nesterov looks ahead before correcting.

Vectors carry activations, losses create gradients, and optimizers turn gradients into parameter movement. The same local update can help or overshoot when repeated across a real training run. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def split_scale(X, y):
    if len(y) > 300:
        x_small, _, y_small, _ = train_test_split(X, y, train_size=300, random_state=6, stratify=y)
    else:
        x_small = X
        y_small = y
    x_tr, x_te, y_tr, y_te = train_test_split(x_small, y_small, test_size=0.4, random_state=0, stratify=y_small)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def one_hot(y, classes):
    out = np.zeros((len(y), classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / np.sum(ez, axis=1, keepdims=True)


def relu(z):
    return np.maximum(z, 0.0)


def init_weights(n_in, n_hidden, n_out, mode, seed):
    rng = np.random.default_rng(seed)
    if mode == "xavier":
        scale1 = math.sqrt(2.0 / (n_in + n_hidden))
        scale2 = math.sqrt(2.0 / (n_hidden + n_out))
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "he":
        scale1 = math.sqrt(2.0 / n_in)
        scale2 = math.sqrt(2.0 / n_hidden)
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "tiny":
        W1 = rng.normal(0.0, 0.01, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.01, size=(n_hidden, n_out))
    elif mode == "large":
        W1 = rng.normal(0.0, 2.0, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 2.0, size=(n_hidden, n_out))
    elif mode == "orthogonal":
        Q1, _ = np.linalg.qr(rng.normal(size=(n_in, max(n_in, n_hidden))))
        Q2, _ = np.linalg.qr(rng.normal(size=(n_hidden, max(n_hidden, n_out))))
        W1 = Q1[:, :n_hidden]
        W2 = Q2[:, :n_out]
    else:
        W1 = rng.normal(0.0, 0.1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.1, size=(n_hidden, n_out))
    b1 = np.zeros(n_hidden)
    b2 = np.zeros(n_out)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}


def forward(params, X, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    W1 = params["W1"]
    if dropconnect_p > 0.0 and rng is not None:
        keep_w = 1.0 - dropconnect_p
        mask_w = rng.binomial(1, keep_w, size=W1.shape) / keep_w
        W1 = W1 * mask_w
    z1 = X @ W1 + params["b1"]
    h1 = relu(z1)
    mask = None
    if dropout_p > 0.0 and rng is not None:
        keep = 1.0 - dropout_p
        mask = rng.binomial(1, keep, size=h1.shape) / keep
        h1 = h1 * mask
    logits = h1 @ params["W2"] + params["b2"]
    return z1, h1, logits, mask


def loss_and_grads(params, X, y, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    classes = params["b2"].shape[0]
    z1, h1, logits, mask = forward(params, X, dropout_p, rng, dropconnect_p)
    probs = softmax(logits)
    target = one_hot(y, classes)
    loss = -np.mean(np.sum(target * np.log(probs + 1e-12), axis=1))
    dlogits = (probs - target) / len(y)
    dW2 = h1.T @ dlogits
    db2 = np.sum(dlogits, axis=0)
    dh1 = dlogits @ params["W2"].T
    if mask is not None:
        dh1 = dh1 * mask
    dz1 = dh1 * (z1 > 0.0)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0)
    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    return loss, grads


def predict(params, X):
    _, _, logits, _ = forward(params, X)
    return np.argmax(logits, axis=1)


def eval_loss(params, X, y):
    classes = params["b2"].shape[0]
    _, _, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    return -float(np.mean(np.sum(target * np.log(probs + 1e-12), axis=1)))


def vector_norm(params):
    total = 0.0
    for value in params.values():
        total += float(np.sum(value * value))
    return math.sqrt(total)


def kfac_precondition_grads(params, X, y, grads, damping):
    classes = params["b2"].shape[0]
    z1, h1, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    dlogits = (probs - target) / len(y)
    dh1 = dlogits @ params["W2"].T
    dz1 = dh1 * (z1 > 0.0)
    out = {key: value.copy() for key, value in grads.items()}
    A1 = X.T @ X / len(y) + damping * np.eye(X.shape[1])
    S1 = dz1.T @ dz1 / len(y) + damping * np.eye(dz1.shape[1])
    A2 = h1.T @ h1 / len(y) + damping * np.eye(h1.shape[1])
    S2 = dlogits.T @ dlogits / len(y) + damping * np.eye(dlogits.shape[1])
    out["W1"] = np.linalg.solve(A1, grads["W1"]) @ np.linalg.inv(S1)
    out["W2"] = np.linalg.solve(A2, grads["W2"]) @ np.linalg.inv(S2)
    return out


def apply_update(params, grads, state, method, lr, t, config):
    beta1 = config.get("beta1", 0.9)
    beta2 = config.get("beta2", 0.999)
    eps = config.get("eps", 1e-8)
    mu = config.get("momentum", 0.0)
    weight_decay = config.get("weight_decay", 0.0)
    for key in params:
        grad = grads[key]
        if method == "sgd":
            update = -lr * grad
        elif method == "momentum":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        elif method == "adagrad":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc += grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "rmsprop":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc *= beta2
            acc += (1.0 - beta2) * grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "adam" or method == "adamw":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            m_hat = m / (1.0 - beta1 ** t)
            v_hat = v / (1.0 - beta2 ** t)
            update = -lr * m_hat / (np.sqrt(v_hat) + eps)
            if method == "adamw" and key.startswith("W"):
                update -= lr * weight_decay * params[key]
        elif method == "lion":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            blended = beta1 * m + (1.0 - beta1) * grad
            update = -lr * np.sign(blended)
            m *= beta2
            m += (1.0 - beta2) * grad
        elif method == "lamb":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            raw = m / (np.sqrt(v) + eps)
            if key.startswith("W"):
                raw += weight_decay * params[key]
            ratio = np.linalg.norm(params[key]) / (np.linalg.norm(raw) + eps)
            ratio = float(np.clip(ratio, 0.1, 10.0))
            update = -lr * ratio * raw
        elif method == "nesterov":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        else:
            update = -lr * grad
        if weight_decay > 0.0 and method not in ["adamw", "lamb"] and key.startswith("W"):
            update -= lr * weight_decay * params[key]
        params[key] += update


def train_mlp(x_tr, y_tr, x_te, y_te, method="sgd", init="he", epochs=12, lr=0.05, hidden=8, batch_size=None, config=None, dropout_p=0.0, dropconnect_p=0.0, seed=0, early_patience=None):
    if config is None:
        config = {}
    classes = int(np.max(y_tr)) + 1
    params = init_weights(x_tr.shape[1], hidden, classes, init, seed)
    state = {}
    rng = np.random.default_rng(seed + 100)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "norm": []}
    best_loss = float("inf")
    best_params = None
    bad_epochs = 0
    n = len(y_tr)
    if batch_size is None:
        batch_size = n
    for epoch in range(1, epochs + 1):
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            if method == "nesterov":
                lookahead = {}
                for key in params:
                    velocity = state.setdefault("v_" + key, np.zeros_like(params[key]))
                    lookahead[key] = params[key].copy()
                    params[key] += config.get("momentum", 0.9) * velocity
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
                for key in params:
                    params[key] = lookahead[key]
            else:
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
            if method == "kfac":
                grads = kfac_precondition_grads(params, x_tr[idx], y_tr[idx], grads, config.get("damping", 0.03))
                apply_update(params, grads, state, "sgd", lr, epoch, config)
            else:
                apply_update(params, grads, state, method, lr, epoch, config)
        train_loss = eval_loss(params, x_tr, y_tr)
        val_loss = eval_loss(params, x_te, y_te)
        train_acc = accuracy_score(y_tr, predict(params, x_tr))
        val_acc = accuracy_score(y_te, predict(params, x_te))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["norm"].append(vector_norm(params))
        if val_loss < best_loss:
            best_loss = val_loss
            best_params = {key: value.copy() for key, value in params.items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if early_patience is not None and bad_epochs >= early_patience:
            params = best_params
            break
    return params, history


def run_component_ladder(variants, metric="accuracy", epochs=12, hidden=16):
    rows = []
    histories = {}
    artifacts = {}
    for rung_index, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
        x_tr, x_te, y_tr, y_te = split_scale(X, y)
        histories[name] = {}
        artifacts[name] = {}
        for variant in variants:
            params, hist = train_mlp(
                x_tr,
                y_tr,
                x_te,
                y_te,
                method=variant.get("method", "sgd"),
                init=variant.get("init", "he"),
                epochs=variant.get("epochs", epochs),
                lr=variant.get("lr", 0.05),
                hidden=hidden,
                batch_size=variant.get("batch_size"),
                config=variant.get("config", {}),
                dropout_p=variant.get("dropout_p", 0.0),
                dropconnect_p=variant.get("dropconnect_p", 0.0),
                seed=variant.get("seed", 10 + rung_index),
                early_patience=variant.get("early_patience"),
            )
            preds = predict(params, x_te)
            acc = accuracy_score(y_te, preds)
            val_loss = eval_loss(params, x_te, y_te)
            value = acc if metric == "accuracy" else val_loss
            rows.append({"rung": name, "variant": variant["name"], "accuracy": acc, "loss": val_loss, "metric": value})
            histories[name][variant["name"]] = hist
            artifacts[name][variant["name"]] = (x_te, y_te, preds)
    return rows, histories, artifacts


def print_table(rows, metric_name):
    print(f"{'rung':34s} {'variant':18s} {metric_name:>10s} {'acc':>8s} {'loss':>8s}")
    for row in rows:
        print(f"{row['rung'][:34]:34s} {row['variant'][:18]:18s} {row['metric']:10.3f} {row['accuracy']:8.3f} {row['loss']:8.3f}")


def plot_results(rows, histories, artifacts, metric_name, best_variant):
    rung_names = list(histories.keys())
    fig, axes = plt.subplots(2, len(rung_names), figsize=(3.2 * len(rung_names), 6.4))
    for col, rung in enumerate(rung_names):
        x_te, y_te, preds = artifacts[rung][best_variant]
        if x_te.shape[1] > 2:
            shown = PCA(n_components=2, random_state=0).fit_transform(x_te)
        else:
            shown = x_te[:, :2]
        axes[0, col].scatter(shown[:, 0], shown[:, 1], c=preds, s=12, cmap="tab10", alpha=0.85)
        axes[0, col].set_title(rung.split("(")[0].strip())
        axes[0, col].set_xticks([])
        axes[0, col].set_yticks([])
        hist = histories[rung][best_variant]
        curve_key = "val_acc" if metric_name == "accuracy" else "val_loss"
        axes[1, col].plot(hist[curve_key], label=best_variant)
        axes[1, col].set_xlabel("epoch")
        axes[1, col].set_title(metric_name)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 3.5))
    variants = sorted({row["variant"] for row in rows})
    for variant in variants:
        vals = [row["metric"] for row in rows if row["variant"] == variant]
        ax.plot(range(1, len(vals) + 1), vals, marker="o", label=variant)
    ax.set_xticks(range(1, len(rung_names) + 1))
    ax.set_xticklabels([f"D{i}" for i in range(1, len(rung_names) + 1)])
    ax.set_ylabel(metric_name)
    ax.set_title("Same ladder, component varied")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## The concept, built once: velocity

The lesson formula is
$$v_t=\mu v_{t-1}-\eta g_t,\qquad \theta_t=\theta_{t-1}+v_t.$$
We first reproduce the scalar training-knob number from the lesson: $\theta=2.000$, $\eta=0.070$, $g=1.200$.

In [ ]:

def optimizer_sweep(momentum):
    theta = 2.0
    eta = 0.070
    gradient = 1.200
    velocity = 0.0
    velocity = momentum * velocity - eta * gradient
    theta = theta + velocity
    return theta, velocity

plain_theta, plain_velocity = optimizer_sweep(0.0)
print(plain_theta, plain_velocity)
assert abs(plain_theta - 1.916) < 1e-12
assert abs(plain_velocity + 0.084) < 1e-12


Nesterov evaluates the gradient at a lookahead location before applying the same velocity memory. The code below keeps that lookahead explicit rather than pretending it is ordinary SGD.

In [ ]:

def nesterov_scalar(theta, velocity, eta, momentum, gradient_fn):
    lookahead = theta + momentum * velocity
    gradient = gradient_fn(lookahead)
    new_velocity = momentum * velocity - eta * gradient
    new_theta = theta + new_velocity
    return new_theta, new_velocity, lookahead

new_theta, new_velocity, lookahead = nesterov_scalar(2.0, -0.084, 0.070, 0.9, lambda value: 1.2)
print(round(lookahead, 4), round(new_theta, 4), round(new_velocity, 4))
assert abs(lookahead - 1.9244) < 1e-12
assert abs(new_theta - 1.8404) < 1e-12


## The dataset ladder

Every topic uses the same `clf_digits_ladder()` and the same small MLP. Only the named optimizer or regularization component changes from variant to variant.

In [ ]:

rungs = clf_digits_ladder()
for name, X, y in rungs:
    classes = np.unique(y)
    print(f"{name:38s} shape={X.shape} classes={len(classes)} sample_y={y[:8].tolist()}")
print("D1 sample X:")
print(rungs[0][1])


## Run the same method across D1-D5

The architecture, splits, scaling, and seed policy stay fixed. The table reports one comparable metric per rung.

In [ ]:

variants = [
    {"name": "SGD", "method": "sgd", "lr": 0.05},
    {"name": "momentum", "method": "momentum", "lr": 0.04, "config": {"momentum": 0.9}},
    {"name": "Nesterov", "method": "nesterov", "lr": 0.03, "config": {"momentum": 0.9}},
]

rows, histories, artifacts = run_component_ladder(variants, metric="accuracy", epochs=10, hidden=8)
print_table(rows, "accuracy")


## Results visualization

Top row: small multiples of held-out predictions. Bottom row: validation curves for the highlighted variant, followed by the component summary curve.

In [ ]:

plot_results(rows, histories, artifacts, "accuracy", "momentum")


## Pitfall on D5: local improvement is not global learning

A larger momentum learning rate can overshoot on noisy D5. Lowering the step or using the lookahead-style setting makes the validation curve less erratic.

In [ ]:

name, X, y = clf_digits_ladder()[-1]
x_tr, x_te, y_tr, y_te = split_scale(X, y)
_, bad = train_mlp(x_tr, y_tr, x_te, y_te, method="momentum", lr=0.12, epochs=10, config={"momentum": 0.95}, seed=77)
_, fixed = train_mlp(x_tr, y_tr, x_te, y_te, method="momentum", lr=0.03, epochs=10, config={"momentum": 0.9}, seed=77)
print("overshoot final val acc", round(bad["val_acc"][-1], 3), "best", round(max(bad["val_acc"]), 3))
print("fixed final val acc", round(fixed["val_acc"][-1], 3), "best", round(max(fixed["val_acc"]), 3))
assert np.isfinite(fixed["val_acc"][-1])


## Evaluate it + Practice

- Main metric: held-out accuracy on every D1-D5 rung, compared with a no-skill baseline near random guessing.
- Sanity check: D1 XOR should improve above chance once the hidden ReLU layer is active.
- Ablation: set momentum to zero or raise the learning rate until the velocity overshoots; the metric should drop or the curve should become less stable.
- Failure signals: exploding loss, flat accuracy near chance, or a D5 train/validation gap that moves in opposite directions.
- Reproducibility: seeds are fixed and the ladder uses sklearn-bundled data only.

Practice 1: Change one hyperparameter in the strongest variant and rerun the summary curve.

Practice 2: Add a new diagnostic printout that distinguishes train accuracy from validation accuracy.

Practice 3: Explain why D5 is harder than D1 using the table and one plotted curve.